# 8장 다중 에이전트 협업

여러 에이전트가 작업을 나누어 수행합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 8장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1 - 언어 모델 선택

이번 실습에 사용할 언어 모델을 지정합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B(내장 추론과 도구 호출 지원)
llm = LLM(model="gemma4:e4b", think=True)

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. 에이전트를 도구로 사용하기

하위 에이전트의 수가 늘면 어느 에이전트에 어떤 작업을 맡길지 결정해야 합니다. 여기서는 중앙의 조정 에이전트 하나가 다른 에이전트에 작업을 지시하는 방식을 사용합니다. 먼저 계산을 담당할 `MathAgent`를 만들고 조정 에이전트가 호출할 수 있는 도구로 등록합니다.

In [ ]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import NativeTools
from illustrated_agents.chapters.ch6 import NativeReAct, TinyAgent
from illustrated_agents.toolbox import add, subtract, multiply

# 계산 담당
math_tools = NativeTools()
math_tools.add_tool("add", add)
math_tools.add_tool("subtract", subtract)
math_tools.add_tool("multiply", multiply)

# 계산 에이전트
math_agent = TinyAgent(
    llm=llm, 
    tools=math_tools, 
    memory=Memory(),
    planner=NativeReAct()
)

같은 방식으로 날짜 처리를 담당할 하위 에이전트를 정의합니다.

In [ ]:
import datetime

def today() -> str:
    """Return today's date (YYYY-MM-DD)."""
    return datetime.date.today().isoformat()

def days_between(a: str, b: str) -> int:
    """Days between two ISO dates."""
    return (datetime.date.fromisoformat(b) - datetime.date.fromisoformat(a)).days

# 날짜 담당
date_tools = NativeTools()
date_tools.add_tool("today", today)
date_tools.add_tool("days_between", days_between)

# 날짜 에이전트
date_agent = TinyAgent(
    llm=llm, 
    tools=date_tools, 
    memory=Memory(),
    planner=NativeReAct()
)

## 3. 조정 에이전트(Orchestrator)

In [ ]:
def ask_math_agent(question: str) -> str:
    """Delegate to the math specialist."""
    return math_agent.run(question)

def ask_date_agent(question: str) -> str:
    """Ask queries to a sub-Agent that handles ISO dates."""
    return date_agent.run(question)

In [ ]:
# 계산 에이전트와 날짜 에이전트를 도구로 등록
tools = NativeTools()
tools.add_tool("ask_math_agent", ask_math_agent)
tools.add_tool("ask_date_agent", ask_date_agent)

# 조정 에이전트
orchestrator_agent = TinyAgent(
    llm=llm, 
    tools=tools, 
    memory=Memory(),
    planner=NativeReAct()
)

`orchestrator_agent`는 하위 에이전트를 도구처럼 호출합니다. 이 실습에서는 한 기기에서 여러 모델을 실행할 필요가 없도록 모든 에이전트가 같은 모델을 사용합니다. 계산과 날짜를 처리하는 `math_agent`와 `date_agent`에는 작은 모델을, 작업 분배와 실행 순서를 결정하는 조정 에이전트에는 더 큰 모델을 사용할 수도 있습니다.

간단한 질문을 보내 조정 에이전트가 어떤 작업을 위임하는지 확인합니다.

실습 질문은 “2030년까지 하루에 4유로씩 모으면 얼마인가요?”입니다. 날짜 계산과 금액 계산을 각각 어느 에이전트에 맡기는지 확인합니다.

In [ ]:
orchestrator_agent.run("If I save €4 per day until 2030, how much will I have?")

In [ ]:
for index, step in enumerate(orchestrator_agent.trajectory.runs[0]["steps"]):
    print(f"-- Step {index+1} ---")
    if step.action:
        print(f"Tool: {step.action}")
        print(f"Observation: {step.observation}\n")
    else:
        print(f"Answer: {step.answer}\n")

In [ ]:
for index, step in enumerate(date_agent.trajectory.runs[0]["steps"]):
    print(f"-- Step {index+1} ---")
    if step.action:
        print(f"Tool: {step.action}")
        print(f"Observation: {step.observation}\n")
    else:
        print(f"Answer: {step.answer}\n")

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(orchestrator_agent.trajectory)

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(date_agent.trajectory)

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(orchestrator_agent.trajectory)

---

# 구현한 내용

하위 에이전트를 도구로 등록했습니다. 조정 에이전트는 다른 도구를 부를 때와 같은 방식으로 하위 에이전트에 질문을 전달하고 결과를 받습니다.

# 다음 장

[9장](../chapter09/chapter09.ipynb)에서는 `TinyAgent`가 이미지와 텍스트를 함께 처리하도록 확장합니다.